# Preparation

Extract corporate participants from the transcript and store the names in a variable.

In [ ]:
import re

def extract_corporate_participants(filepath):
    """Return corporate participant names from a transcript TXT file."""
    with open(filepath, "r", encoding="utf-8") as file:
        text = file.read()

    participants_section = re.search(
        r"Corporate Participants\s*=+\n(.*?)\n={3,}", text, re.S
    )
    if not participants_section:
        print("No 'Corporate Participants' section found.")
        return []

    names = re.findall(r"\* (.*?)\n", participants_section.group(1))

    print("Extracted corporate participant names:")
    for name in names:
        print(f"- {name}")

    return names


filepath = "path/to/transcript.txt"
corporate_names = extract_corporate_participants(filepath)


Extract the **Questions and Answers** section and store it in a variable.

In [ ]:
def extract_qa_section(filepath):
    """Return the Questions and Answers section from a transcript TXT file."""
    with open(filepath, "r", encoding="utf-8") as file:
        text = file.read()

    qa_match = re.search(r"Questions and Answers\s*\n-+\n(.*)", text, re.S)
    if not qa_match:
        print("No 'Questions and Answers' section found.")
        return ""

    qa_content = qa_match.group(1)
    print("\nExtracted Q&A section successfully.")
    return qa_content


qa_section = extract_qa_section(filepath)
print(qa_section)


This step stores each participant block in one row. To combine all sentences from the same participant later, use the grouped example below.

In [ ]:
import pandas as pd
import re

def extract_qa_for_participants(qa_content, corporate_names):
    """Collect each Q&A response block spoken by a corporate participant."""
    qa_blocks = re.split(r"\n-+\n", qa_content)
    relevant_data = {name: [] for name in corporate_names}

    for i in range(len(qa_blocks) - 1):
        block = qa_blocks[i].strip()
        next_block = qa_blocks[i + 1].strip()

        lines = block.splitlines()
        if lines:
            speaker_match = re.match(r"^(.*?),", lines[0].strip())
            if speaker_match:
                speaker_name = speaker_match.group(1).strip()
                if speaker_name in corporate_names and next_block:
                    relevant_data[speaker_name].append(next_block)

    return {k: v for k, v in relevant_data.items() if v}


qa_data = extract_qa_for_participants(qa_section, corporate_names)

rows = []
for name, sentences in qa_data.items():
    for sentence in sentences:
        rows.append({"Corporate Participant": name, "Sentences": sentence})

df = pd.DataFrame(rows)
print("\nExtracted Q&A data:")
print(df)

output_path = "path/to/extracted_qa_results.csv"
df.to_csv(output_path, index=False)
print(f"\nData saved to '{output_path}'.")


Extract the **Presentation** section and store it in a variable.

In [ ]:
import re

def extract_presentation_section(filepath):
    """Return the Presentation section from a transcript TXT file."""
    with open(filepath, "r", encoding="utf-8") as file:
        text = file.read()

    presentation_match = re.search(
        r"Presentation\s*\n-+\n(.*?)(?=\n=+\n)", text, re.S
    )
    if not presentation_match:
        print("No 'Presentation' section found.")
        return ""

    presentation_content = presentation_match.group(1).strip()
    print("\nExtracted 'Presentation' section successfully.")
    return presentation_content


presentation_content = extract_presentation_section(filepath)
print("\nPresentation section content:")
print(presentation_content)


This step stores each participant block in one row. You can combine rows for the same participant later if needed.

In [ ]:
import pandas as pd
import re

def extract_presentation_for_participants(presentation_content, corporate_names):
    """Collect each presentation block spoken by a corporate participant."""
    presentation_blocks = re.split(r"\n-+\n", presentation_content)
    relevant_data = {name: [] for name in corporate_names}

    for i in range(len(presentation_blocks) - 1):
        block = presentation_blocks[i].strip()
        next_block = presentation_blocks[i + 1].strip()

        lines = block.splitlines()
        if lines:
            speaker_match = re.match(r"^(.*?),", lines[0].strip())
            if speaker_match:
                speaker_name = speaker_match.group(1).strip()
                if speaker_name in corporate_names and next_block:
                    relevant_data[speaker_name].append(next_block)

    return {k: v for k, v in relevant_data.items() if v}


presentation_data = extract_presentation_for_participants(
    presentation_content, corporate_names
)

rows = []
for name, sentences in presentation_data.items():
    for sentence in sentences:
        rows.append(
            {"Corporate Participant": name, "Presentation Sentences": sentence}
        )

df = pd.DataFrame(rows)
print("\nExtracted presentation data:")
print(df)

output_path = "path/to/extracted_presentation_results.csv"
df.to_csv(output_path, index=False)
print(f"\nData saved to '{output_path}'.")


Extract key metadata from transcript file names.

In [ ]:
import os
import pandas as pd

def process_file_titles(folder_path):
    """Extract metadata from transcript file names in a folder."""
    data = []

    for file_name in os.listdir(folder_path):
        if file_name.endswith("transcript.txt"):
            components = file_name.split("-")
            year = components[0]
            quarter = components[1]
            company_code = components[3]
            numeric_code = components[4].split(".")[0]

            data.append(
                {
                    "Year": year,
                    "Quarter": quarter,
                    "Company Code": company_code,
                    "Numeric Code": numeric_code,
                }
            )

    return pd.DataFrame(data)


folder_path = "path/to/transcript_folder"
extracted_info_df = process_file_titles(folder_path)

print("\nExtracted file information:")
print(extracted_info_df)

output_path = "path/to/extracted_file_info.csv"
extracted_info_df.to_csv(output_path, index=False)
print(f"\nData saved to '{output_path}'.")


# Part I - Extraction Workflow

In [ ]:
import os
import re
import pandas as pd
import pickle

def process_file_titles(folder_path):
    """Extract metadata from transcript file names in a folder."""
    data = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith("transcript.txt"):
            components = file_name.split("-")
            data.append(
                {
                    "File Name": file_name,
                    "Year": components[0],
                    "Quarter": components[1],
                    "Company Code": components[3],
                    "Numeric Code": components[4].split(".")[0],
                }
            )
    return pd.DataFrame(data)


def extract_corporate_participants(filepath):
    """Return corporate participant names from a transcript TXT file."""
    with open(filepath, "r", encoding="utf-8") as file:
        text = file.read()

    participants_section = re.search(
        r"Corporate Participants\s*=+\n(.*?)\n={3,}", text, re.S
    )
    if not participants_section:
        return []

    return re.findall(r"\* (.*?)\n", participants_section.group(1))


def extract_qa_section(filepath):
    """Return the Questions and Answers section from a transcript TXT file."""
    with open(filepath, "r", encoding="utf-8") as file:
        text = file.read()

    qa_match = re.search(r"Questions and Answers\s*\n-+\n(.*)", text, re.S)
    return qa_match.group(1) if qa_match else ""


def extract_qa_for_participants(qa_content, corporate_names):
    """Collect each Q&A response block spoken by a corporate participant."""
    qa_blocks = re.split(r"\n-+\n", qa_content)
    relevant_data = {name: [] for name in corporate_names}

    for i in range(len(qa_blocks) - 1):
        block = qa_blocks[i].strip()
        next_block = qa_blocks[i + 1].strip()
        lines = block.splitlines()

        if lines:
            speaker_match = re.match(r"^(.*?),", lines[0].strip())
            if speaker_match and speaker_match.group(1).strip() in corporate_names:
                relevant_data[speaker_match.group(1).strip()].append(next_block)

    return relevant_data


def extract_presentation_section(filepath):
    """Return the Presentation section from a transcript TXT file."""
    with open(filepath, "r", encoding="utf-8") as file:
        text = file.read()

    presentation_match = re.search(
        r"Presentation\s*\n-+\n(.*?)(?=\n={3,})", text, re.S
    )
    return presentation_match.group(1).strip() if presentation_match else ""


def extract_presentation_for_participants(presentation_content, corporate_names):
    """Collect each presentation block spoken by a corporate participant."""
    presentation_blocks = re.split(r"\n-+\n", presentation_content)
    relevant_data = {name: [] for name in corporate_names}

    for i in range(len(presentation_blocks) - 1):
        block = presentation_blocks[i].strip()
        next_block = presentation_blocks[i + 1].strip()
        lines = block.splitlines()

        if lines:
            speaker_match = re.match(r"^(.*?),", lines[0].strip())
            if speaker_match and speaker_match.group(1).strip() in corporate_names:
                relevant_data[speaker_match.group(1).strip()].append(next_block)

    return relevant_data


def process_txt_folder(folder_path, output_path):
    """Process transcript files and save combined participant-level outputs."""
    file_info_df = process_file_titles(folder_path)
    final_data = []

    for _, row in file_info_df.iterrows():
        file_name = row["File Name"]
        filepath = os.path.join(folder_path, file_name)

        corporate_names = extract_corporate_participants(filepath)
        qa_content = extract_qa_section(filepath)
        qa_data = extract_qa_for_participants(qa_content, corporate_names)
        presentation_content = extract_presentation_section(filepath)
        presentation_data = extract_presentation_for_participants(
            presentation_content, corporate_names
        )

        for name in corporate_names:
            final_data.append(
                {
                    "File Name": file_name,
                    "Year": row["Year"],
                    "Quarter": row["Quarter"],
                    "Company Code": row["Company Code"],
                    "Numeric Code": row["Numeric Code"],
                    "Corporate Participant": name,
                    "Q&A Sentences": " ".join(qa_data.get(name, [])),
                    "Presentation Sentences": " ".join(presentation_data.get(name, [])),
                }
            )

    final_df = pd.DataFrame(final_data)

    pickle_path = os.path.join(output_path, "processed_data.pkl")
    final_df.to_pickle(pickle_path)
    print(f"\nProcessed data saved to '{pickle_path}'.")

    csv_path = os.path.join(output_path, "processed_data.csv")
    final_df.to_csv(csv_path, index=False)
    print(f"\nProcessed data saved to CSV at '{csv_path}'.")

    return final_df


folder_path = "path/to/input_folder"
output_path = "path/to/output_folder"
os.makedirs(output_path, exist_ok=True)
final_df = process_txt_folder(folder_path, output_path)

print("\nSample of final processed data:")
print(final_df.head())


In [ ]:
import re
from nltk.tokenize import RegexpTokenizer

def load_stop_words(file_path):
    """Load stop words from a TXT file."""
    with open(file_path, "r", encoding="utf-8") as file:
        return set(line.strip().lower() for line in file)


stop_words_file = "path/to/stop_words.txt"
stop_words = load_stop_words(stop_words_file)

print(f"Loaded {len(stop_words)} stop words.")
print(stop_words)


def clean_text(content):
    """Lowercase text, keep hyphenated words, and remove stop words."""
    content = content.lower()
    content = re.sub(r"[^a-z\s\-]", " ", content)
    tokenizer = RegexpTokenizer(r"\b\w[\w-]*\b")
    tokens = tokenizer.tokenize(content)
    cleaned_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(cleaned_tokens)


# Part II

In [ ]:
import os
import re
from collections import Counter
import dask.dataframe as dd
import pandas as pd
from nltk.tokenize import RegexpTokenizer

# Step 1 is merged into parse_file_name() below.

def extract_corporate_participants(file_content):
    """Return corporate participant names from transcript text."""
    participants_section = re.search(
        r"Corporate Participants\s*=+\n(.*?)\n={3,}", file_content, re.S
    )
    if not participants_section:
        return []
    return re.findall(r"\* (.*?)\n", participants_section.group(1))


def extract_qa_section(file_content):
    """Return the Questions and Answers section from transcript text."""
    qa_match = re.search(r"Questions and Answers\s*\n-+\n(.*)", file_content, re.S)
    return qa_match.group(1) if qa_match else ""


def extract_qa_for_participants(qa_content, corporate_names):
    """Collect each Q&A response block spoken by a corporate participant."""
    qa_blocks = re.split(r"\n-+\n", qa_content)
    relevant_data = {name: [] for name in corporate_names}

    for i in range(len(qa_blocks) - 1):
        block = qa_blocks[i].strip()
        next_block = qa_blocks[i + 1].strip()
        lines = block.splitlines()

        if lines:
            speaker_match = re.match(r"^(.*?),", lines[0].strip())
            if speaker_match and speaker_match.group(1).strip() in corporate_names:
                relevant_data[speaker_match.group(1).strip()].append(next_block)

    return relevant_data


def extract_presentation_section(file_content):
    """Return the Presentation section from transcript text."""
    has_qa_section = re.search(r"Questions and Answers\s*\n-+\n", file_content, re.S)

    if has_qa_section:
        presentation_match = re.search(
            r"Presentation\s*\n-+\n(.*?)(?=\n={3,})", file_content, re.S
        )
        return presentation_match.group(1).strip() if presentation_match else ""

    fallback_match = re.search(r"Presentation\s*\n-+\n(.*)", file_content, re.S)
    return fallback_match.group(1).strip() if fallback_match else ""


def extract_presentation_for_participants(presentation_content, corporate_names):
    """Collect each presentation block spoken by a corporate participant."""
    presentation_blocks = re.split(r"\n-+\n", presentation_content)
    relevant_data = {name: [] for name in corporate_names}

    for i in range(len(presentation_blocks) - 1):
        block = presentation_blocks[i].strip()
        next_block = presentation_blocks[i + 1].strip()
        lines = block.splitlines()

        if lines:
            speaker_match = re.match(r"^(.*?),", lines[0].strip())
            if speaker_match and speaker_match.group(1).strip() in corporate_names:
                relevant_data[speaker_match.group(1).strip()].append(next_block)

    return relevant_data


def clean_text(content, stop_words):
    """Lowercase text, keep hyphenated words, and remove stop words."""
    content = content.lower()
    content = re.sub(r"[^a-z\s\-]", " ", content)
    tokenizer = RegexpTokenizer(r"\b\w[\w-]*\b")
    tokens = tokenizer.tokenize(content)
    cleaned_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(cleaned_tokens)


def tokenize_and_count(data, stop_words):
    """Clean each participant's text blocks and count words."""
    tokenized_data = {}
    total_word_count = {}

    for participant, text_blocks in data.items():
        cleaned_text = " ".join(clean_text(block, stop_words) for block in text_blocks)
        tokenized_data[participant] = cleaned_text
        total_word_count[participant] = len(cleaned_text.split())

    return tokenized_data, total_word_count


def count_lexicon_occurrences(text, lexicon):
    """Count lexicon terms in text without extra category filters."""
    count = Counter()
    sorted_lexicon = sorted(lexicon, key=len, reverse=True)

    for term in sorted_lexicon:
        term_pattern = re.escape(term)
        term_regex = re.compile(rf"(?<!\w){term_pattern}(?![\w\-])", re.IGNORECASE)
        matches = term_regex.findall(text)
        count[term] += len(matches)

    return count


def parse_file_name(file_name):
    """Extract metadata from a transcript file name."""
    components = file_name.split("-")
    return {
        "File Name": file_name,
        "Year": components[0],
        "Quarter": components[1],
        "Company Code": components[3],
        "Numeric Code": components[4].split(".")[0],
    }


def process_and_save_results(folder_path, lexicon, stop_words, output_file=None):
    """Process transcript files and save lexicon-based results."""
    results = []
    total_files = len(
        [f for f in os.listdir(folder_path) if f.lower().endswith("transcript.txt")]
    )
    processed_files = 0

    for file_name in os.listdir(folder_path):
        if file_name.lower().endswith("transcript.txt"):
            processed_files += 1
            print(f"Processing file {processed_files}/{total_files}: {file_name}")

            metadata = parse_file_name(file_name)
            file_path = os.path.join(folder_path, file_name)

            with open(file_path, "r", encoding="utf-8") as file:
                file_content = file.read()

            participants = extract_corporate_participants(file_content)
            qa_section = extract_qa_section(file_content)
            presentation_section = extract_presentation_section(file_content)
            metadata["Corporate Participants"] = participants

            qa_cleaned_text = clean_text(qa_section, stop_words)
            qa_word_count = len(qa_cleaned_text.split())

            presentation_cleaned_text = clean_text(
                presentation_section, stop_words
            )
            presentation_word_count = len(presentation_cleaned_text.split())

            qa_lexicon_counts = count_lexicon_occurrences(qa_cleaned_text, lexicon)
            presentation_lexicon_counts = count_lexicon_occurrences(
                presentation_cleaned_text, lexicon
            )

            qa_lexicon_proportion = (
                sum(qa_lexicon_counts.values()) / qa_word_count
                if qa_word_count > 0
                else "Missing"
            )
            presentation_lexicon_proportion = (
                sum(presentation_lexicon_counts.values()) / presentation_word_count
                if presentation_word_count > 0
                else "Missing"
            )

            row_result = {
                **metadata,
                "Q&A Total Word Count": qa_word_count,
                "Presentation Total Word Count": presentation_word_count,
                "QA Lexicon Proportion": qa_lexicon_proportion,
                "Presentation Lexicon Proportion": presentation_lexicon_proportion,
            }
            results.append(row_result)

    final_df = dd.from_pandas(pd.DataFrame(results), npartitions=10)

    output_base = os.path.basename(folder_path.rstrip(os.sep))
    csv_output_file = f"{output_base}_results.csv"
    pickle_output_file = f"{output_base}_results.pkl"

    final_df.compute().to_csv(csv_output_file, index=False)
    print(f"Results saved as CSV: {csv_output_file}")

    final_df.compute().to_pickle(pickle_output_file)
    print(f"Results saved as Pickle: {pickle_output_file}")


def load_txt_file(file_path):
    """Load a TXT file into a lowercase set."""
    with open(file_path, "r", encoding="utf-8") as file:
        return set(line.strip().lower() for line in file)


In [ ]:
def main():
    """Run the transcript-processing workflow."""
    folder_path = "path/to/transcript_folder"
    stop_words_file = "path/to/stop_words.txt"
    lexicon_file = "path/to/lexicon.txt"

    stop_words = load_txt_file(stop_words_file)
    lexicon = load_txt_file(lexicon_file)

    print(f"Stop words loaded: {len(stop_words)}")
    print(f"Lexicon words loaded: {len(lexicon)}")

    process_and_save_results(
        folder_path=folder_path,
        lexicon=lexicon,
        stop_words=stop_words,
    )


if __name__ == "__main__":
    main()
